# Baselines + Validation Strategy

In [2]:
# Imports + Load + Preprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from src.preprocessing import PreprocessConfig, preprocess_train_test

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 180)

TRAIN_PATH = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Train.csv"
TEST_PATH  = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

cfg = PreprocessConfig(id_col="ID", target_col="Target")

train, test = preprocess_train_test(train_raw, test_raw, cfg, for_model="lightgbm")  # fill missing categoricals with "missing"
TARGET = cfg.target_col
ID = cfg.id_col

print("train:", train.shape, "| test:", test.shape)
train[[ID, TARGET]].head()

train: (9618, 47) | test: (2405, 46)


,ID,Target
0,ID_3CFL0U,Low
1,ID_XWI7G3,Medium
2,ID_TY93LV,Low
3,ID_9OP2C8,Low
4,ID_13REYS,Low


In [3]:
# Prepare X/y and identify feature types

y = train[TARGET]
X = train.drop(columns=[TARGET])

#Identify categorical columns/strings and numeric columns
cat_cols = X.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("categorical columns:", len(cat_cols))
print("numeric columns:", len(num_cols))
print("example categorical columns:", cat_cols[:10])
print("example numeric columns:", num_cols[:10])

categorical columns: 32
numeric columns: 14
example categorical columns: ['ID', 'country', 'attitude_stable_business_environment', 'attitude_worried_shutdown', 'compliance_income_tax', 'perception_insurance_doesnt_cover_losses', 'perception_cannot_afford_insurance', 'motor_vehicle_insurance', 'has_mobile_money', 'current_problem_cash_flow']
example numeric columns: ['owner_age', 'personal_income', 'business_expenses', 'business_turnover', 'business_age_years', 'business_age_months', 'personal_income_missing', 'business_expenses_missing', 'business_turnover_missing', 'log_personal_income']


## 1) Validation strategy

**We will implement two validation schemes and report both.**

✅ Scheme 1: StratifiedKFold (main CV)

✅ Scheme 2: Group-based diagnostic by country

✅ Scheme 3: Leave-One-Country-Out stress test

In [4]:
# Define evaluation helpers (reusable functions)

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro")

def run_cv(model, X, y, cv_splits, fit_params=None, verbose=False):
    """
    Generic CV runner: fits model on each split, evaluates macro F1.
    """
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(cv_splits, 1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        if fit_params is None:
            model.fit(X_tr, y_tr)
        else:
            model.fit(X_tr, y_tr, **fit_params)

        pred = model.predict(X_va)
        score = macro_f1(y_va, pred)
        scores.append(score)

        if verbose:
            print(f"Fold {fold}: macro F1 = {score:.4f}")

    return np.mean(scores), np.std(scores), scores

In [5]:
# Validation scheme 1: StratifiedKFold (main).
# Keeps class proportions balanced in each fold.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_splits = list(skf.split(X, y))

print("StratifiedKFold splits:", len(skf_splits))

StratifiedKFold splits: 5


In [6]:
# Validation scheme 2: GroupKFold by country (diagnostic)
# checks if our model generalizes across countries.

if "country" in X.columns:
    groups = X["country"]
    gkf = GroupKFold(n_splits=4)  # 4 countries -> 4 folds (each fold leaves out one country)
    gkf_splits = list(gkf.split(X, y, groups=groups))
    print("GroupKFold splits:", len(gkf_splits))
else:
    gkf_splits = None
    print("No country column found; skipping GroupKFold.")

GroupKFold splits: 4


In [7]:
# Validation scheme 3: Leave-One-Country-Out (LOCO) stress test
# basically the same idea as GroupKFold but we explicitly report performance per held-out country.


def leave_one_country_out_splits(X, y, country_col="country"):
    splits = []
    for c in sorted(X[country_col].dropna().unique()):
        va_idx = X.index[X[country_col] == c].to_numpy()
        tr_idx = X.index[X[country_col] != c].to_numpy()
        splits.append((tr_idx, va_idx, c))
    return splits

loco_splits = None
if "country" in X.columns:
    loco_splits = leave_one_country_out_splits(X, y, "country")
    print("LOCO folds:", len(loco_splits))

LOCO folds: 4


## 2) Baselines

We will build:

- Baseline A: Majority class

- Baseline B: Logistic Regression (OneHotEncoder)

- Baseline C: ExtraTrees (or RandomForest)

And produce A results table comparing CV scores under StratifiedKFold and GroupKFold.

In [8]:
# Baseline A — Majority Class (sanity check)

from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent", random_state=42)

mean_f1, std_f1, _ = run_cv(dummy, X, y, skf_splits, verbose=False)
print(f"Dummy (Most Frequent) - StratifiedKFold macro F1: {mean_f1:.4f} ± {std_f1:.4f}")

if gkf_splits is not None:
    mean_f1_g, std_f1_g, _ = run_cv(dummy, X, y, gkf_splits, verbose=False)
    print(f"Dummy (Most Frequent) - GroupKFold macro F1: {mean_f1_g:.4f} ± {std_f1_g:.4f}")

Dummy (Most Frequent) - StratifiedKFold macro F1: 0.2633 ± 0.0000
Dummy (Most Frequent) - GroupKFold macro F1: 0.2619 ± 0.0266


In [10]:
# Baseline B — Logistic Regression (OneHotEncoder)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# scale data
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))  # with_mean=False keeps compatibility with sparse matrices
])

# Preprocess for logistic regression
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
])

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", cat_transformer, cat_cols),
        ("num", num_transformer, num_cols)
    ],
    remainder="drop"
)

logreg = LogisticRegression(
    max_iter=10000,
    class_weight="balanced",
    solver="saga",
    C=0.5   # smaller C => stronger regularization
)

logreg_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", logreg)
])

mean_f1, std_f1, _ = run_cv(logreg_model, X, y, skf_splits)
print(f"LogReg (OHE) - StratifiedKFold macro F1: {mean_f1:.4f} ± {std_f1:.4f}")

if gkf_splits is not None:
    mean_f1_g, std_f1_g, _ = run_cv(logreg_model, X, y, gkf_splits)
    print(f"LogReg (OHE) - GroupKFold macro F1: {mean_f1_g:.4f} ± {std_f1_g:.4f}")

LogReg (OHE) - StratifiedKFold macro F1: 0.0922 ± 0.0307
LogReg (OHE) - GroupKFold macro F1: 0.0635 ± 0.0306


Linear models performed poorly under proper convergence `(macro F1 ≈ 0.09)`, indicating strong nonlinear interactions and complex feature relationships. This motivated the use of tree-based ensemble models better suited to high-cardinality categorical data.

In [11]:
# Baseline C — ExtraTrees (fast & strong tree baseline)

from sklearn.ensemble import ExtraTreesClassifier

# For tree models, we need numeric-only features.
# We'll encode categoricals as integer codes (simple baseline).
X_tree = X.copy()

for c in cat_cols:
    X_tree[c] = X_tree[c].astype("category").cat.codes

# Replace -1 codes (NaN category) with -1 explicitly
X_tree[cat_cols] = X_tree[cat_cols].replace(-1, -1)

et = ExtraTreesClassifier(
    n_estimators=600,
    random_state=42,
    n_jobs=-1,
    max_depth=None,
    class_weight="balanced_subsample"
)

mean_f1, std_f1, _ = run_cv(et, X_tree, y, skf_splits)
print(f"ExtraTrees - StratifiedKFold macro F1: {mean_f1:.4f} ± {std_f1:.4f}")

if gkf_splits is not None:
    mean_f1_g, std_f1_g, _ = run_cv(et, X_tree, y, gkf_splits)
    print(f"ExtraTrees - GroupKFold macro F1: {mean_f1_g:.4f} ± {std_f1_g:.4f}")

ExtraTrees - StratifiedKFold macro F1: 0.7790 ± 0.0168
ExtraTrees - GroupKFold macro F1: 0.4086 ± 0.1555


ExtraTrees achieved strong performance under stratified CV `(macro F1 ≈ 0.78)`, demonstrating substantial nonlinear signal in the features. However, performance dropped significantly under country-grouped validation `(macro F1 ≈ 0.41)`, revealing strong country-specific patterns and potential domain shift. This highlights the importance of group-aware validation and motivates further modeling with algorithms that better capture robust cross-country structure.

In [ ]:
# Produce a baseline results table


In [ ]:
# LOCO (Leave-One-Country-Out) score breakdown


In [ ]:
# Single-fold diagnostics: confusion matrix + classification report
